In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from pathlib import Path
import sys
from bisect import bisect_left, bisect_right
from dataclasses import asdict, dataclass
from typing import Callable
from datetime import datetime
from contextlib import contextmanager
import time
import uuid

root_dir = Path(os.getcwd()).parent
src_dir = root_dir / 'ingestion/src'

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from helpers.pdfparser import PdfParser, MarkdownSection, SectionMetadata
from helpers.indexer import Indexer, IndexDocument, IndexChunk
from helpers.hasher import DocumentHasher
from helpers.dbrepository import FileRepository


os.environ['NLTK_DATA'] = str(root_dir / '.nltk_data')
os.environ['QDRANT_URL'] = 'http://localhost:6333'

/home/chins/git/rag-architecture/ingestion/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def enrich_chunk_with_citation_data(chunk: IndexChunk, metadata: dict, page_offsets: list) -> IndexChunk:

    if chunk and metadata:
        section_start_idx = metadata['section_start_char_idx'] 

        chunk_start_idx = section_start_idx + chunk.start_char_idx
        chunk_end_idx = chunk_start_idx + len(chunk.text) 
    
        chunk_start_page_idx = bisect_right(page_offsets, chunk_start_idx) - 1
        chunk_end_page_idx = bisect_left(page_offsets, chunk_end_idx) - 1

        #set citation data in the chunk object
        chunk.start_char_idx = chunk_start_idx
        chunk.end_char_idx = chunk_end_idx
        chunk.start_page_idx = chunk_start_page_idx
        chunk.end_page_idx = chunk_end_page_idx

    return chunk 

@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        end = time.perf_counter()
        print(f"{label}: {end - start:.6f} seconds")


In [6]:
hasher = DocumentHasher()
file_repository = FileRepository(db_path=str(root_dir / 'database/rag.db'))

file_names = ['2505.07891.pdf', 'OTC_TCS_2025.pdf']

for file_name in file_names:
    print(f'------------ processing {file_name}')
    file_id = str(uuid.uuid4().hex) 
    file_path = root_dir / f'staging/{file_name}'
    file_hash = hasher.hash_document(file_path)
    
    with timer('parsing completed..'):
        parser = PdfParser()
        file_metadata, page_offsets, sections = parser.parse(file_path)    

    with timer('prepare indexing documents..'):
        indexer = Indexer(qdrant_url='http://localhost:6333', collection_name='test_collection2', fast_embedding_name='BAAI/bge-small-en-v1.5')

        #create index_docs
        index_docs = [
                IndexDocument(    
                    doc_id = f'{file_hash}_{hasher.hash_document(section.text.encode())}',
                    text=section.text,
                    metadata=section.metadata,
                ) for section in sections if isinstance(section, MarkdownSection)]

    with timer('indexing completed..'):                                         
        #indexer chunks the documents and indexes them in the vector store.
        index = indexer.index(
            index_docs, 
            transform_chunk_fn=lambda c, m: enrich_chunk_with_citation_data(c, m, page_offsets),
            )

    with timer('state updated in DB..'):
        file_repository.insert_or_ignore(
            file_id=file_id,
            content_hash=file_hash,
            content_length=file_metadata.get('file_length', 0),
            metadata=file_metadata
        )
    

db_path = /home/chins/git/rag-architecture/database/rag.db
------------ processing 2505.07891.pdf
parsing completed..: 17.159446 seconds
prepare indexing documents..: 0.695346 seconds
splitting start: 2026-09-11 11:18:00.420406
splitting end: 2026-09-11 11:18:00.542195
32 nodes created by splitter
32 new node to be indexed
indexing completed..: 49.180229 seconds
state updated in DB..: 0.019050 seconds
------------ processing OTC_TCS_2025.pdf
parsing completed..: 399.842699 seconds
prepare indexing documents..: 0.462210 seconds
splitting start: 2026-09-11 11:25:29.989981
splitting end: 2026-09-11 11:25:30.843580
400 nodes created by splitter
400 new node to be indexed
indexing completed..: 565.227138 seconds
state updated in DB..: 0.028641 seconds


In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.embeddings.fastembed import FastEmbedEmbedding

#initialize local embedding model
local_embed = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
import qdrant_client

# Initialize the Qdrant client
client = qdrant_client.QdrantClient(url="http://localhost:6333")

#initiaalize qdrant vector store
vector_store = QdrantVectorStore(
    client=client, 
    collection_name="test_collection"
    )

# #initialize qdrant vector store with hybrid search enabled
# vector_store = QdrantVectorStore(
#     client=client, 
#     collection_name="hybrid_test_collection",
#     enable_hybrid=True,
#     batch_size=64
#     )

#initialize storage_context over the vector storage
storage_context = StorageContext.from_defaults(vector_store=vector_store)


In [ ]:

async def search_documents(query: str) -> str:
    """Useful for answering natural language questions about individuals"""
    response = await query_engine.aquery(query)
    return str(response)


# Create an enhanced workflow with both tools
agent = FunctionAgent(
    tools=[search_documents],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="""You are a helpful assistant that can search through documents to answer questions. 
    Do not answer questions that are not related to the documents. If you cannot find the answer in the documents, say "I don't know".""",
)

# Now we can ask questions about the documents or do calculations
async def main():
    response = await agent.run(
        "What are the differences between the play and the drama?"
    )
    print(response)

await main()

In [ ]:
client.retrieve('test_collection', ids=['0001dc07-2689-4944-adfb-96f8dbc7eb46'])

In [ ]:
from qdrant_client.http import models


#nodes_to_check = [node for node in nodes]
node_ids_to_check = ['0001dc07-2689-4944-adfb-96f8dbc7eb46']

scroll_results, _ = client.scroll(
    collection_name="test_collection",
    scroll_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="chunk_id", # LlamaIndex maps node_id to the 'id' key in payload
                match=models.MatchAny(any=node_ids_to_check),
            )
        ]
    ),
    with_vectors=False,
    with_payload=True, # We need the payload to read the custom string ID
    limit=len(node_ids_to_check),
)

# 3. Extract the found IDs from the payload
existing_ids = {point.payload["chunk_id"] for point in scroll_results if "chunk_id" in point.payload}

existing_ids

In [ ]:
from dataclasses import dataclass
from markdown_it import MarkdownIt
from bisect import bisect_left, bisect_right
from collections import defaultdict

@dataclass
class SectionMetadata:

    section_title: str = ''
    section_start_char_idx: int = 0 #index in pdf document text where the section starts
    section_end_char_idx: int = 0 #index in pdf document text where the section ends
    
    pdf_file_name: str = '' 
    pdf_file_path: str = ''
    pdf_title: str = ''
    pdf_author: str = ''
    pdf_length: int = 0 #no of characters in pdf file
    pdf_total_pages: int = 0 #total pages in the parent document
    pdf_page_offsets: list = None #pdf page offset positions w.r.t characters 
    document_id: str = ''
    
@dataclass
class MarkdownSection:
    level: int
    text: str
    
    metadata: SectionMetadata

@dataclass
class MarkdownChunk:
    text: str

    chunk_id: str = ''
    start_char_idx: int = 0 #start index of chunk in the original pdf file character stream
    end_char_idx: int = 0 #end index of chunk in the original pdf file character stream
    start_page_idx: int = 0 #pdf page number index where the chunk starts
    end_page_idx: int = 0 #pdf page number index where the chunk ends

    section: MarkdownSection = None

'''
returns start character indexes for each page (0-based). 
'''
def build_page_offsets(pages):
    offsets = []
    start = 0

    for i, page in enumerate(pages):
        offsets.append(start)
        start = start + len(page['text'])
    return offsets

'''
returns pdf file level metadata and page_offsets (mapping between pages and their start character positions)
'''
def get_pdf_metadata(pages):
    metadata = defaultdict(None)
    keys = ['title', 'author', 'file_path', 'page_count']
    if pages and 'metadata' in pages[0]:
        obj = pages[0]['metadata']
        metadata = defaultdict(None, {key : obj[key] for key in keys if key in obj})

    page_offsets = build_page_offsets(pages)
    return metadata, page_offsets

'''
returns index for each new line character (0 based) 
Note that if there are 'm' newline characters (\n), then we have m+1 lines.
so offsets[i] = character index of ith line (0 based) 
'''
def build_line_offsets(text):
    
    offsets = [0] #first line always start at 0th index

    for i, char in enumerate(text):
        if char == "\n":
            offsets.append(i)

    return offsets

'''
takes markdown and and returns the list of heirarchical sections
'''
def markdown_sections(markdown, page_offsets, file_metadata):

    md = MarkdownIt("commonmark")
    tokens = md.parse(markdown)
    line_offsets = build_line_offsets(markdown)
    headings = []

    # pass 1: extract each heading (often heirarchical), its level and start line
    for i, token in enumerate(tokens):

        if token.type != "heading_open":
            continue

        level = int(token.tag[1]) #e.g for h1, level=1

        # <heading_open> <inline> <heading_close>
        inline_token = tokens[i + 1]
        section_title = inline_token.content
        start_line = token.map[0]

        headings.append({
            "level": level,
            "section_title": section_title,
            "start_line": start_line, 
        })

    # pass 2: create sections from the headings. A section defined by a heading at level l ends when we encounter the 
    # first heading with level <= l (or the end of text)
    # note that start_line, start_idx are inclusive, end_line, end_idx are exclusive
    sections = []
    n = len(headings)

    for i in range(n): #for each heading index
        heading = headings[i]
        level = heading["level"]
        start_line = heading["start_line"]

        # Find the next heading with the same or lower level. That ends the current section
        end_line = len(line_offsets)
        for j in range(i+1, n):
            next_heading = headings[j]
            if next_heading["level"] <= level: #new section start detected. it's start is the end of the current section
                end_line = next_heading["start_line"]
                break

        #map the section start and end lines to start and end character positions 
        start_idx = line_offsets[start_line]

        if end_line < len(line_offsets):
            end_idx = line_offsets[end_line]
        else:
            end_idx = len(markdown)

        sections.append(
            MarkdownSection(
                level=level,
                text=markdown[start_idx:end_idx],

                metadata=SectionMetadata(
                    section_title=heading["section_title"],
                    section_start_char_idx = start_idx,
                    section_end_char_idx = end_idx,

                    pdf_title=file_metadata.get('title', ''), 
                    pdf_author=file_metadata.get('author', ''), 
                    pdf_file_path=file_metadata.get('file_path', ''), 
                    pdf_total_pages=file_metadata.get('page_count', ''), 
                    pdf_length=len(markdown),
                    pdf_page_offsets=page_offsets
                    
                    )
            )
        )
        
    return sections


'''
Takes the hierarchical sections and the cut_level and return a list of non-overlapping and collectively exhausting flat sections.
All sections at level > cut_level are simply absorbed in their parent sections.
For all the sections at level < cut_level (which are parent sections), their spans are adjusted so that they end right where
the first child under them begins resulting in non-overlapping sections.
'''
def get_flat_sections(sections, cut_level=2): 

    root = sections[0]
    markdown = root.text

    #remove (i.e merge with their parent) all sections at finer level than the cut level 
    flat_sections = [section for section in sections if section.level <= cut_level]

    #update end range and text for each parent section
    n = len(flat_sections)

    #process sections from end to start. set start_page and end_page for the last section
    next_section = flat_sections[-1]
    
    #process from second last to first sections
    for i in range(n-1)[::-1]: 
        curr_section = flat_sections[i]

        #if curr section is parent of next section, adjust current section's span so that it ends where the next section begins
        if curr_section.level < next_section.level: 

            curr_section.metadata.section_end_char_idx = next_section.metadata.section_start_char_idx

            #if this is the first section, just take everything from the start
            #this is to avoid extra \n at the beginning of the document
            if i == 0: 
                curr_section.text = markdown[: curr_section.metadata.section_end_char_idx]
            else:
                curr_section.text = markdown[curr_section.metadata.section_start_char_idx: curr_section.metadata.section_end_char_idx]

        

        #set curr as the next section for the next iteration
        next_section = curr_section
        
    return flat_sections

In [ ]:
from llama_index.core.schema import NodeRelationship

'''
Returns chunk object for a given llamaindex chunk
'''
def get_chunk(node):

    source_doc = node.relationships[NodeRelationship.SOURCE]
    parent_section = source_doc.metadata
    page_offsets = parent_section['pdf_page_offsets']
    section_start_idx = parent_section['section_start_char_idx'] 

    chunk_start_idx = section_start_idx + node.start_char_idx
    chunk_end_idx = chunk_start_idx + len(node.text) 

    chunk_start_page_idx = bisect_right(page_offsets, chunk_start_idx) - 1
    chunk_end_page_idx = bisect_left(page_offsets, chunk_end_idx) - 1
     
    return MarkdownChunk(
            text = node.text,

            chunk_id = node.id_,
            start_char_idx = chunk_start_idx,
            end_char_idx= chunk_end_idx,
            start_page_idx=chunk_start_page_idx,
            end_page_idx=chunk_end_page_idx,

            section = parent_section,
            
            )

def update_node_metadata(node, chunk):

    #remove unwanted fields
    for key in ['pdf_page_offsets']:
        if key in node.metadata:
            node.metadata.pop(key)
            
    node.metadata = {
        **node.metadata,
        'chunk_id': node.id_,
        'chunk_start_char_idx': chunk.start_char_idx,
        'chunk_end_char_idx': chunk.end_char_idx,
        'chunk_start_page_idx': chunk.start_page_idx,
        'chunk_end_page_idx': chunk.end_page_idx,
        }

In [ ]:
import pymupdf4llm
from markdown_it import MarkdownIt
from llama_index.core import Document

#read full_text as well as
pages = pymupdf4llm.to_markdown('../staging/2505.07891.pdf', page_chunks=True, force_ocr=False)
full_md = ''.join([page['text'] for page in pages])


In [ ]:
pymupdf4llm.use_layout(False)
pages = pymupdf4llm.to_markdown('../staging/2505.07891.pdf', page_chunks=True,use_ocr=False)
 

In [ ]:
pymupdf4llm.to_text('../staging/2505.07891.pdf')

In [ ]:
import pymupdf

In [ ]:
file_metadata, page_offsets = get_pdf_metadata(pages)

#parse the markdown to find logical sections (treating h2 as the cut_level)
hierarchical_sections = markdown_sections(full_md, page_offsets, file_metadata)
sections = get_flat_sections(hierarchical_sections)

In [ ]:
from dataclasses import asdict
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

excluded_llm_keys = []
excluded_embed_keys = ['pdf_page_offsets', 'pdf_total_pages']

#build documents corresponding to each section
documents = [Document(
    text=section.text, 
    metadata=asdict(section.metadata),

    excluded_embed_metadata_keys=excluded_embed_keys, 
    excluded_llm_metadata_keys=excluded_llm_keys,

    ) for section in sections]


#create chunks (nodes)
splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=128,
)
nodes = splitter.get_nodes_from_documents(documents)

#build chunk objects for citation support
chunks = []
for node in nodes:
    chunk = get_chunk(node)
    update_node_metadata(node, chunk)
    chunks.append(chunk)

#build index over nodes
index = VectorStoreIndex(
    nodes, 
    embed_model=local_embed, 
    storage_context=storage_context
    )



In [ ]:
# for chunk in chunks:
#     print(f'---------------------------------------------------------')
    
#     print(f'chunk span:{chunk.chunk_start_char_idx}-{chunk.chunk_end_char_idx}, pages:{chunk.chunk_start_page}-{chunk.chunk_end_page}')
#     print(f'parent section span:{chunk.section['section_start_char_idx']}-{chunk.section['section_end_char_idx']}')


In [ ]:
# from llama_index.core.schema import NodeRelationship
# node_idx = 10

# chunk = nodes[node_idx]
# source_doc = chunk.relationships[NodeRelationship.SOURCE]
# chunk_start_idx, chunk_end_idx = chunk.start_char_idx, chunk.end_char_idx

# print(source_doc.metadata['start_page'])
# print(chunk_start_idx, chunk_end_idx)

In [7]:
from pydantic import BaseModel, Field

class Citation(BaseModel):
    ''' A citation source'''
    id: int = Field(description="source #")

class AnswerPart(BaseModel):
    '''Answer part backed by citations'''
    part: str =  Field(description="The answer part text")
    citations: list[Citation] = Field(description="A list of citations backing the answer part")

class RAGResponse(BaseModel):
    '''Citation backed answer. An Answer consists of one or more parts'''
    parts: list[AnswerPart] = Field(description="A list of answer parts")


In [9]:
from llama_index.core.query_engine import CitationQueryEngine
from llama_index.core import PromptTemplate
from llama_index.llms.openai import OpenAI

custom_citation_prompt = PromptTemplate(
    "Please answer the query based on the provided context.\n"
    "Every time you use a source text to generate your answer part, you must add the corresponding citation source.\n"
    
    "Do not add explicit references in square brackets in your answer. Instead use the citations in the structured output resposne."
    "Do not use external knowledge. If you do not know the answer, send empty response.\n\n"
    "Context:\n{context_str}\n\n"
    "Query: {query_str}\n\n"
    "Answer:"
)

llm = OpenAI(model="gpt-4o-mini").as_structured_llm(RAGResponse)

query_engine = CitationQueryEngine.from_args(
    index,
    llm=llm,
    citation_chunk_size=512,       # Granular text blocks for inline citations
    citation_chunk_overlap=20,     # Small overlap to prevent cut-off quotes
    similarity_top_k=3,            # Number of source nodes to retrieve
    citation_qa_template=custom_citation_prompt,
)

retriever = index.as_retriever()

In [10]:
response = query_engine.query('Explain the topic specific text-rank in short. Also, what is the benefit of TrumourGPT?')
response

PydanticResponse(response=RAGResponse(parts=[AnswerPart(part='Topic-specific TextRank (TST) is an adaptation of the traditional TextRank algorithm that prioritizes sentences based on their relevance to a specific topic. It modifies the conventional TextRank by incorporating topic relevance scores, which measure how closely a sentence relates to the target topic. This adaptation involves changes in vertex selection, edge weighting, and the scoring algorithm to emphasize thematic elements, particularly in health-related content. TST enhances the extraction of key sentences by assigning higher weights to those that are more relevant to the identified topics.', citations=[Citation(id=1)]), AnswerPart(part='TrumorGPT benefits from using TST by efficiently constructing semantic health knowledge graphs from extensive user inputs. It leverages few-shot learning to adapt quickly from a small number of examples, allowing it to identify and extract key phrases and sentences that represent the cor

In [20]:
response = query_engine.query('What does the letter of TCS chairman N Chandrashekhar say?')
response


PydanticResponse(response=RAGResponse(parts=[AnswerPart(part="In the letter from the Chairman, N Chandrasekaran, he expresses gratitude for the continued support and trust of stakeholders in TCS's journey. He highlights the appointment of seasoned leaders, Aarthi and Mangesh, who are expected to play a crucial role in enhancing the company's growth and operational excellence. The letter emphasizes TCS's commitment to delivering value and setting benchmarks in the IT services industry, focusing on innovation and exploring new technologies to stay ahead in a changing landscape.", citations=[Citation(id=1)])]), source_nodes=[NodeWithScore(node=TextNode(id_='756bb7c1-cc8c-4062-a83a-f209259a555d', embedding=None, metadata={'section_title': '**Letter from the Chairman**', 'section_start_char_idx': 11354, 'section_end_char_idx': 28476, 'file_name': 'OTC_TCS_2025.pdf', 'file_path': '/home/chins/git/rag-architecture/staging/OTC_TCS_2025.pdf', 'file_title': 'TCS Integrated Annual Report 2024-25 

In [21]:
for source in response.source_nodes:
    print(source.node.text)


Source 1:
Both Aarthi and Mangesh are seasoned, respected leaders whose expertise and vision will play a pivotal role in strengthening our growth journey, refining our organizational structure, and enhancing our operational excellence. 

We remain steadfast in our commitment to delivering value to all stakeholders and setting new benchmarks in the IT services industry. Our focus on innovation and growth drives us to continuously explore new technologies and business models, ensuring we stay ahead in an ever-evolving landscape. 

We thank you for your continued support and trust in our journey. 

Best regards, 

###### **K Krithivasan** 

Chief Executive Officer and Managing Director 



**Integrated Annual Report 2024-25** 

**The Year Gone By** 

**12** 

### **The Year Gone By Q4** 

**Tata Group Chairman N Chandrasekaran** was awarded the UK Knighthood ‘ **Most Excellent Order of the British Empire’,** announced by the UK government, for his services to build UK and India business r

In [ ]:
# from llama_index.core.schema import NodeRelationship
# sources = response.source_nodes

# text1 = sources[0].node.text[10:]
# text2 = sources[1].node.text[10:]

# source_chunk_id = sources[0].metadata['chunk_id']
# source_node = index.docstore.get_node(source_chunk_id)

# print(f'node text: {source_node.text[:100]}......{source_node.text[-100:]}')

# print(f'text1 start: {text1[:100]}')
# print(f'text2 end  : {text2[-100:]}')

# len(text1 + text2), len(source_node.text)


In [ ]:
async def search_documents(query: str) -> str:
    """Useful for answering natural language questions about documents"""
    response = await query_engine.aquery(query)
    return str(response)


# Create an enhanced workflow with both tools
agent = FunctionAgent(
    tools=[search_documents],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="""
    You are a helpful assistant that can search through documents to answer questions. 
    Do not answer questions that are not related to the documents. For provenance, generate citation which includes the details about the source file name, section name, and the psource age(s) 
    which your answers are based on. If you cannot find the answer in the documents, say "I don't know".""",
)

# Now we can ask questions about the documents or do calculations
async def main():
    response = await agent.run(
        "Explain the topic specific text-rank. Also, what are the three techniques used by Trurumour to verify the truthfulness of the health news?"
    )
    print(response)

await main()

In [ ]:
import sqlite3

db_path = "../database/rag.db"

def initialize_db(db_path):
    with sqlite3.connect(db_path) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                document_id TEXT PRIMARY KEY,
                content_hash TEXT NOT NULL UNIQUE,
                file_size INTEGER NOT NULL,
                file_name TEXT NOT NULL,
                file_path TEXT NOT NULL,
                created_at TEXT NOT NULL,
                metadata TEXT
            )
        """)

initialize_db(db_path)

In [ ]:
class DocumentRepository:
    def __init__(self, db_path: str):
        self.db_path = db_path
        self.create_schema(db_path)

    def create_schema(self, db_path):
        with sqlite3.connect(db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS documents (
                    document_id TEXT PRIMARY KEY,
                    content_hash TEXT NOT NULL UNIQUE,
                    file_size INTEGER NOT NULL,
                    file_name TEXT NOT NULL,
                    file_path TEXT NOT NULL,
                    metadata TEXT,
                    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
                )
            """)

    def find_by_hash(self, content_hash):
        with sqlite3.connect(self.db_path) as conn:
            return conn.execute(
                """
                SELECT document_id, file_size
                FROM documents
                WHERE content_hash = ?
                """,
                (content_hash,),
            ).fetchone()

    def insert_or_ignore(self, document_id, content_hash, file_size, file_name, file_path, metadata='{}'):
        ret = None
        with sqlite3.connect(self.db_path) as conn:
            
            ret = conn.execute(
                """
                INSERT OR IGNORE INTO documents
                    (document_id, content_hash, file_size, file_name, file_path, metadata)
                VALUES (?, ?, ?, ?, ?, ?)
                RETURNING document_id;
                """,
                (document_id, content_hash, file_size, file_name, file_path, metadata),
            ).fetchone()
        print(ret)
        return ret

In [ ]:
repository = DocumentRepository(db_path)

repository.insert_or_ignore('123', 'abc', 100, 'agas.txt', '../staging/agas.txt')
repository.insert_or_ignore('124', 'xyz', 200, 'ads.txt', 'http://foo.bar.com/ads.txt')


In [ ]:
repository.find_by_hash('xyzc')

In [ ]:
repository.insert_or_ignore('13ss2', 'abcfs', 100, 'agas.txt', '../staging/agas.txt')

In [ ]:
import hashlib
m = hashlib.sha256()
m.update(b"Nobody inspects")
m.update(b" the spammish repetition")

print(m.hexdigest())

m2 = hashlib.sha256()
m2.update(b"Nobody inspects the spammish repetition")
print(m2.hexdigest())

In [ ]:
arr = ['foo', 'bar']
for i, ele in enumerate(arr):
    print(i, ele)